### Inject chirp signals into existing h5 capture file

Careful: May have problems writing to h5 format.   To be safe, use a copy of the source file to avoid potential for corruption.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

%matplotlib inline
import matplotlib.pyplot as plt
params = {'legend.fontsize': 'medium',
          'figure.figsize': (10,6),
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large'}
plt.rcParams.update(params)

import numpy as np
from astropy import units as u
from astropy.coordinates import Angle
import blimpy as bl
import time
import pandas as pd

import src.plot_fns as pltg             # generic plot fns
import src.plot_h5_psd_sg1 as plt_h5    # blimpy-based plot fns

from pathlib import Path

sys.path.append(os.getenv('SETIGEN_PATH'))

import setigen as stg

import h5py

sg_dir = os.getenv('SGDIR') + '/'
if not os.path.isdir(sg_dir[0:-1]):
    os.system('mkdir '+sg_dir[0:-1])

def db(x):
    """ Convert linear value to dB value """
    return 10*np.log10(np.abs(x.astype(np.float64))+1e-20)

def find(cond,dim=0):
    """ Return indices according to conditon, e.g. find(x>2), like matlab find() """
    return np.nonzero(cond)[dim]

%matplotlib inline

In [ ]:
try:
    parameters_are_undefined
except NameError:
    parameters_are_undefined = True     
    print('Parameters are undefined, using defaults\n')


In [3]:
# import h5py
# h5py.run_tests()

#### Define run parameters

In [ ]:
if parameters_are_undefined:
    # output_ext = '.fil'
    # use .fil output if problems occur writing to h5 output, then do fil2h5 afterwards 
    output_ext = '.h5'

    display_figs01 = True
    beep_when_h5_complete = True
    plot_channel_idx_dets = True

    if (0):
        sig_max_drift = 5.0    # Hz/sec
        total_drift_cycles = 10.0
        n_sig_per_cycle = 20.1
        drift_offset = -.02
        sig_snr_db = 20.
        sig_width_bins = 1
        do_snr_compensation = False
    elif (1):
        sig_max_drift = 5.0    # Hz/sec
        total_drift_cycles = 10.0
        n_sig_per_cycle = 20.1
        drift_offset = -.02
        sig_snr_db = 15.
        sig_width_bins = 1
        do_snr_compensation = True
    
    sig_min_drift = -sig_max_drift
    src_h5_name = '' 

    fig_dir = 'plots/'

    if (1):
        src_h5_name = 'synthetic_chi2_pfb0_0dB_8ch.h5'
        # src_h5_name = 'synthetic_chi2_pfb0_2dB_8ch.h5'
        # src_h5_name = 'synthetic_chi2_pfb8_0dB_8ch.h5'
        
        # src_h5_name = 'blc71_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'
        # src_h5_name = 'blc72_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'
        # src_h5_name = 'blc73_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'
        # src_h5_name = 'blc74_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'
        # src_h5_name = 'blc75_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'
        # src_h5_name = 'blc76_guppi_58832_16209_MESSIER031_0057.gpuspec.0000.h5'

        # src_h5_name = 'blc00_guppi_59239_37260_HIP50422_0024.rawspec.0000.h5'
        # src_h5_name = 'blc01_guppi_59239_37260_HIP50422_0024.rawspec.0000.h5'
        # src_h5_name = 'blc11_guppi_59189_50683_TIC36724087_0022.rawspec.0000.h5'
        # src_h5_name = 'blc12_guppi_59189_50683_TIC36724087_0022.rawspec.0000.h5'
        # src_h5_name = 'blc20_guppi_59239_37260_HIP50422_0024.rawspec.0000.h5'
        # src_h5_name = 'blc47_guppi_59103_03394_DIAG_HIP95631_0015.rawspec.0000.h5'
        
        

full_src_h5_name = sg_dir + src_h5_name
full_src_fil_name = full_src_h5_name.split('.h5')[0] + '.fil'


#### Get source background file parameters and determine chirp targets to inject

In [ ]:
import src.get_h5_info as h5info
p = h5info.get_h5_params(full_src_h5_name)

h5_size_MB =p['h5_size_MB']
fig_f_limits_MHz = [p['f_min_MHz'],p['f_max_MHz']]

fine_fft_size = p['fine_fft_size']
n_lti = p['n_lti']
n_sti = p['n_sti']
n_avg = p['n_avg']
f_min_MHz = p['f_min_MHz']
f_max_MHz = p['f_max_MHz']
t_obs = p['t_obs']
fch1 = p['fch1']
foff = p['foff']

telescope = p['telescop']
h5_size_MB = p['h5_size_MB']
n_coarse_channels = p['n_coarse_channels']

# open directory for figures if necessary
if not os.path.isdir(fig_dir[0:-1]):
    os.system('mkdir '+fig_dir[0:-1])

n_inject = round(total_drift_cycles*n_sig_per_cycle)
offset_index = fine_fft_size/8
global_index1 = offset_index
global_index2 = n_coarse_channels*fine_fft_size - offset_index
global_inject_index = np.linspace(global_index1,global_index2,n_inject)
global_index_delta = (global_inject_index[1]-global_inject_index[0])
global_index_mod_fft = np.mod(global_inject_index,fine_fft_size)

f_inject1_MHz = fch1 + global_index1*foff
f_inject2_MHz = fch1 + global_index2*foff
f_inject_MHz = fch1 + global_inject_index*foff
f_delta_KHz = (f_inject_MHz[1]-f_inject_MHz[0])*1e3

drift_inject = np.sign(foff)*sig_max_drift*((np.mod(np.linspace(0.5,total_drift_cycles+.5,n_inject),1)-0.5)*2.-drift_offset)

unit_drift_rate = p['fs_fine']*p['fs_fine']/p['n_sti']
smeared_bins = np.ceil(np.abs(drift_inject) / unit_drift_rate).astype(int)
smeared_bins[smeared_bins==0] = 1

if (do_snr_compensation):
    snr_inject_db = sig_snr_db*np.ones(n_inject) + 5.*np.log10(smeared_bins)
    snr_comp_string = 'c'
else:
    snr_inject_db = sig_snr_db*np.ones(n_inject)
    snr_comp_string = ''

snr_inject_linear = (10**(.1*snr_inject_db))*np.ones(n_inject)
    
smeared_snr_db = snr_inject_db - 10.*np.log10(smeared_bins)
desmeared_snr_db = snr_inject_db - 5.*np.log10(smeared_bins)

sig_width_Hz = sig_width_bins*p['fs_fine']
    
max_drift_MHz = sig_max_drift*t_obs*1e-6

config_str = f'{n_inject} Injected Signals, Drift:±{sig_max_drift:.1f} Hz/sec, {sig_snr_db} dB, {sig_width_Hz:.0f} Hz BW'
config_str2 = f'-{n_inject}sig-{sig_max_drift:.0f}Hzpersec-{snr_comp_string}{sig_snr_db:.0f}dB-{sig_width_Hz:.0f}Hz'
print(config_str,'\n',config_str2)

f_start1_MHz = p['ctr_freq_MHz'] - 2*p['fs_coarse']*1e-6
f_stop1_MHz = p['ctr_freq_MHz'] + 2*p['fs_coarse']*1e-6
fig_f_limits_MHz = [f_start1_MHz,f_stop1_MHz]

# new_fil_name = src_h5_name.split('.h5')[0] + config_str2 + '_test.fil'
new_fil_name = src_h5_name.split('.h5')[0] + config_str2 + '.fil'
full_new_fil_name = sg_dir + new_fil_name
print(f'{new_fil_name = }')

# new_h5_name = src_h5_name.split('.h5')[0] + config_str2 + '_test.h5'
new_h5_name = src_h5_name.split('.h5')[0] + config_str2 + '.h5'
full_new_h5_name = sg_dir + new_h5_name
print(f'{new_h5_name = }')

npz_name = src_h5_name.split('.h5')[0] + config_str2 + '.npz'
full_npz_name = sg_dir + npz_name  

fig_name_base =  src_h5_name.split('.h5')[0] + config_str2
print(f'{fig_name_base = }')


In [ ]:
%tb

In [ ]:
print(drift_inject)

In [ ]:
print(max(drift_inject),min(drift_inject))

In [ ]:
ii = find(abs(drift_inject)<.01)
print(ii)
print(drift_inject[ii])


In [ ]:
print(f'{global_index1=}, {global_index2=}, {global_index_delta=:.1f}')
print(f'{f_inject1_MHz=:.3f}, {f_inject2_MHz=:.3f}, {f_delta_KHz=:.3f}')

In [11]:

# os.system('h52fil '+full_src_h5_name+' -n '+full_src_fil_name)
# frame.save_fil(full_src_fil_name)
# from blimpy import Waterfall
# wf = Waterfall(full_src_fil_name)
# wf.write_to_hdf5(h5_file_name)

In [ ]:
ii = find(abs(drift_inject)<.01)
print(ii)
print(drift_inject[ii])


#### Plot out locations of injected chirp targets

In [ ]:
for i_ptype in range(3):

    f1_MHz = f_min_MHz
    f2_MHz = f_max_MHz
    drift1 = sig_min_drift*1.5
    drift2 = sig_max_drift*1.5
    line_width1 = 1
    line_width2 = 1
    x_limits=[f1_MHz,f2_MHz]
    x_label = 'Frequency (MHz)'
    if (i_ptype==0):
        fig_id = '01-'
        x_data1=[f_inject_MHz]
        y_data1=[drift_inject]
        xy_markers1 = ['g*']
        xy_legend1 = ['Injected Drift']
        x_data2=[f_inject_MHz]
        y_data2=[snr_inject_db]
        xy_markers2 = ['g*']
        xy_legend2 = ['Injected SNR']
    elif (i_ptype==1):
        if (plot_channel_idx_dets):
            fig_id = '02-'
            x_limits=[0.,1.]
            x_data1=[global_index_mod_fft/fine_fft_size]
            y_data1=[drift_inject]
            xy_markers1 = ['g*']
            xy_legend1 = ['Injected Drift']
            x_data2=[global_index_mod_fft/fine_fft_size]
            y_data2=[snr_inject_db]
            xy_markers2 = ['g*']
            xy_legend2 = ['Injected SNR']
            x_label = f"Normalized Coarse Channel Index, chan_bw={p['chan_bw']*1e-6:.3f} MHz"
    elif (i_ptype==2):
        fig_id = '03-'
        x_data1=[f_inject_MHz]
        y_data1=[drift_inject]
        xy_markers1 = ['g*']
        xy_legend1 = ['Injected Drift']
        x_data2=[f_inject_MHz,f_inject_MHz,f_inject_MHz]
        y_data2=[snr_inject_db,smeared_snr_db,desmeared_snr_db]
        xy_markers2 = ['g*','r*','b*']
        xy_legend2 = ['Injected SNR','Expected SNR After Smearing','Expected SNR After De-Smearing']
    else:
        continue    # skip for now
    
    %matplotlib inline
    fig = plt.figure(figsize=(10, 6))

    plt.subplot(2,1,1)

    fig_text_list1=[[.15,.84,new_h5_name],
            [.15,.81,f'{f_min_MHz:.0f}-{f_max_MHz:.0f} MHz, {sig_min_drift:.0f}->{sig_max_drift:.0f} Hz/sec, {t_obs:.1f} sec']]
    
    # plot drift rates on linear scale, pos & neg values
    pltg.plot_generic(fig=fig,x_data=x_data1,y_data=y_data1,xy_markers=xy_markers1,line_width=line_width1,xy_legend=xy_legend1,
            x_limits=x_limits,
            y_limits=[drift1,drift2],
            x_label = [],
            y_label = 'Drift Rate (Hz/sec)',
            fig_title= telescope.upper() + ' Injected Chirps: ' + config_str,
            fig_text_list=fig_text_list1)
            
    plt.subplot(2,1,2)

    fig_text_list2=[[.15,.43,f'h5 {h5_size_MB:3.0f} MB, {n_coarse_channels} Coarse Chnl']]
    
    pltg.plot_generic(fig=fig,x_data=x_data2,y_data=y_data2,xy_markers=xy_markers2,line_width=line_width2,xy_legend=xy_legend2,
                x_limits=x_limits,
                y_limits=[0., 40.],
                x_label = x_label,
                y_label = 'SNR(dB)',
                fig_title= '',
                fig_text_list=fig_text_list2,
                legend_loc = 'upper right')

    plt.savefig(fig_dir + fig_id + fig_name_base+'-inj.png',bbox_inches='tight')

    if display_figs01:
        plt.show()
    else:
        plt.close(fig)


#### Load h5 source file and convert to setigen frame

In [14]:
wf = bl.Waterfall(full_src_h5_name)
# wf.info()
# wf.write_to_fil(full_new_fil_name)

c = stg.Frame(waterfall=wf)
freqs, _ = wf.grab_data()

#### Determine background levels as a function of frequency

#### Do the signal injections via setigen

In [ ]:
whos

In [ ]:
p

In [ ]:
noise_mean_inject = np.zeros(n_inject)
noise_std_inject = np.zeros(n_inject)


for i in range(n_inject):

    # define bounding frequencies

    if (drift_inject[i]>0):
        f_bound1 = f_inject_MHz[i] - max_drift_MHz*0.1
        f_bound2 = f_inject_MHz[i] + max_drift_MHz*1.1
    elif (drift_inject[i]<0):
        f_bound1 = f_inject_MHz[i] - max_drift_MHz*1.1
        f_bound2 = f_inject_MHz[i] + max_drift_MHz*0.1
    else:
        f_bound1 = f_inject_MHz[i] - max_drift_MHz*0.5
        f_bound2 = f_inject_MHz[i] + max_drift_MHz*0.5
    
    # if i%11 == 0:
    #     print(f'{i+1:4d} of {n_inject}, {drift_inject[i]:6.2f} Hz/sec, {f_inject_MHz[i]:.3f} MHz, {f_bound1:.4f}-{f_bound2:.4f}')

    data_window = bl.Waterfall(full_src_h5_name, f_bound1, f_bound2)
    subc = stg.Frame(data_window)
    
    noise_mean_inject[i],noise_std_inject[i] = subc.get_noise_stats()
    
    drift_value = drift_inject[i]
    sig_level=subc.get_intensity(snr=snr_inject_linear[i])
    smearing_subsamples = smeared_bins[i]
    
    if (i%11 == 0)|(i<n_inject/10):
        print(f'{i+1:4d} of {n_inject}, {drift_inject[i]:6.2f} Hz/sec, {f_inject_MHz[i]:.3f} MHz, {drift_value=:.2f}, {sig_level=:.0f}, {sig_width_Hz=:.2f}, {unit_drift_rate=:.2f}, {smearing_subsamples=} ')


    sig = c.add_signal( path = stg.constant_path(f_start=(f_inject_MHz[i]*u.MHz),
                        drift_rate=drift_inject[i]*u.Hz/u.s),
                        t_profile = stg.constant_t_profile(level=sig_level),
                        f_profile = stg.box_f_profile(width=sig_width_Hz*u.Hz),
                        bp_profile = stg.constant_bp_profile(level=1),
                        bounding_f_range=(f_bound1*u.MHz, f_bound2*u.MHz),
                        doppler_smearing=True,
                        smearing_subsamples=smearing_subsamples)
    


In [ ]:
print(f'{full_new_fil_name=}')
print(f'{full_new_h5_name=}')

In [ ]:
if (1):
    if (output_ext=='.fil'):
        c.save_fil(full_new_fil_name)
    else:
        # problems have occurred here in some cases
        c.save_h5(full_new_h5_name)
else:
    c.save_fil(full_new_fil_name)
    if (output_ext=='.h5'):
        # problems have occurred here in some cases
        os.system('fil2h5 ' + full_new_fil_name + ' -n ' + full_new_h5_name)



#### Add truth information for injected signals into output h5 file

In [ ]:
if (output_ext=='.h5'):
    full_new_h5_temp_name = full_new_h5_name.split('.h5')[0] + config_str2 + '_temp.h5'
    os.system('cp '+full_new_h5_name+' '+full_new_h5_temp_name)
    print(f'Writing injected signal information to h5 file, setigen group')
    with h5py.File(full_new_h5_temp_name, 'a') as f:
        # Create the setigen group
        grp = f.create_group("setigen")
        # Create datasets within the group
        grp.create_dataset("start_frequency", data=f_inject_MHz)
        grp.create_dataset("drift_rate", data=drift_inject)
        grp.create_dataset("zscore", data=snr_inject_linear)
        grp.create_dataset("signal_width", data=smeared_bins)
        grp.create_dataset("snr_db", data=snr_inject_db)
        grp.create_dataset("noise_mean", data=noise_mean_inject)
        grp.create_dataset("noise_std", data=noise_std_inject)
        grp.create_dataset("n_inject", data=n_inject)
    # new h5 file can balloon in size, need to repack to expected size
    os.system('h5repack '+full_new_h5_temp_name+' '+full_new_h5_name)
    os.system('rm '+full_new_h5_temp_name)

if (1):
    # Write the data to a np savez file
    print(f'Writing injected signal information to {full_npz_name}')
    np.savez_compressed(full_npz_name,
                        start_frequency=f_inject_MHz,
                        drift_rate=drift_inject,
                        zscore=snr_inject_linear,
                        signal_width=smeared_bins,
                        snr_db=snr_inject_db,
                        noise_mean=noise_mean_inject,
                        noise_std=noise_std_inject,
                        n_inject=n_inject)



full_h5_repack_name

In [20]:
if beep_when_h5_complete:
    try:
        # Beep in WSL
        if os.system("powershell.exe '[console]::beep(261.6,700)'") !=0:
            raise Exception('powershell.exe not found')
    except:
        # linux, probably doesn't work
        print('Beep!')
        os.system("echo -ne '\a'")